## Cell 1

In [1]:
import os
import cv2
import shutil
import hashlib
from PIL import Image
from collections import Counter
import matplotlib.pyplot as plt
import numpy as np
import warnings
warnings.filterwarnings('ignore')

## Cell 2 - Configuration

In [3]:
# ⚠️ Change these paths to match yours
TRAIN_DIR = r'D:\New folder\data\train'
TEST_DIR  = r'D:\New folder\data\test'

# Cleaning settings
MIN_SIZE         = 50
MIN_FILE_SIZE    = 1000
BLUR_THRESHOLD   = 50
DARK_THRESHOLD   = 20
BRIGHT_THRESHOLD = 240
VALID_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff'}

# Show current dataset
for split, path in [('TRAIN', TRAIN_DIR), ('TEST', TEST_DIR)]:
    print(f'\n  {split}:')
    for cls in os.listdir(path):
        cls_path = os.path.join(path, cls)
        if os.path.isdir(cls_path):
            print(f'    {cls}: {len(os.listdir(cls_path))} images')



  TRAIN:
    cataract: 1431 images
    normal: 1191 images

  TEST:
    cataract: 359 images
    normal: 300 images


## Cell 3 - Inspect Dataset 

In [5]:
def get_image_hash(fpath):
    with open(fpath, 'rb') as f:
        return hashlib.md5(f.read()).hexdigest()

def check_blur(fpath):
    img = cv2.imread(fpath)
    if img is None: return 0
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    return cv2.Laplacian(gray, cv2.CV_64F).var()

def check_brightness(fpath):
    img = cv2.imread(fpath)
    if img is None: return 0
    return cv2.cvtColor(img, cv2.COLOR_BGR2GRAY).mean()

def inspect_dataset(base_dir, remove_bad=False):
    stats = dict(total=0, corrupted=0, duplicates=0,
                 too_small=0, blurry=0, dark=0, bright=0,
                 wrong_ext=0, removed=0)
    seen_hashes = {}
    bad_files = []

    for cls in sorted(os.listdir(base_dir)):
        cls_path = os.path.join(base_dir, cls)
        if not os.path.isdir(cls_path): continue
        for fname in os.listdir(cls_path):
            fpath = os.path.join(cls_path, fname)
            if not os.path.isfile(fpath): continue
            stats['total'] += 1
            reason = None
            ext = os.path.splitext(fname)[1].lower()
            if ext not in VALID_EXTENSIONS:
                reason = f'wrong extension ({ext})'
                stats['wrong_ext'] += 1
            elif os.path.getsize(fpath) < MIN_FILE_SIZE:
                reason = 'file too small'
                stats['too_small'] += 1
            else:
                try:
                    img = Image.open(fpath)
                    img.verify()
                    img = Image.open(fpath)
                    w, h = img.size
                    if w < MIN_SIZE or h < MIN_SIZE:
                        reason = f'too small ({w}x{h})'
                        stats['too_small'] += 1
                    else:
                        img_hash = get_image_hash(fpath)
                        if img_hash in seen_hashes:
                            reason = 'duplicate'
                            stats['duplicates'] += 1
                        else:
                            seen_hashes[img_hash] = fname
                            blur = check_blur(fpath)
                            if blur < BLUR_THRESHOLD:
                                reason = f'blurry (score={blur:.1f})'
                                stats['blurry'] += 1
                            else:
                                br = check_brightness(fpath)
                                if br < DARK_THRESHOLD:
                                    reason = f'too dark ({br:.1f})'
                                    stats['dark'] += 1
                                elif br > BRIGHT_THRESHOLD:
                                    reason = f'too bright ({br:.1f})'
                                    stats['bright'] += 1
                except Exception:
                    reason = 'corrupted'
                    stats['corrupted'] += 1
            if reason:
                bad_files.append((fpath, reason))
                if remove_bad:
                    try:
                        os.remove(fpath)
                        stats['removed'] += 1
                    except Exception as e:
                        print(f'  Cannot remove {fname}: {e}')

    print('=' * 50)
    print(' Cleaning Report:')
    print('=' * 50)
    print(f'  Total images : {stats["total"]}')
    print(f'  Corrupted    : {stats["corrupted"]}')
    print(f'  Duplicates   : {stats["duplicates"]}')
    print(f'  Too small    : {stats["too_small"]}')
    print(f'  Blurry       : {stats["blurry"]}')
    print(f'  Too dark     : {stats["dark"]}')
    print(f'  Too bright   : {stats["bright"]}')
    print(f'  Wrong ext    : {stats["wrong_ext"]}')
    print(f'\n   Total issues : {len(bad_files)}')
    if remove_bad:
        print(f'  Removed      : {stats["removed"]} images')
    return bad_files


print(' TRAIN:')
bad_train = inspect_dataset(TRAIN_DIR, remove_bad=False)
print('\n TEST:')
bad_test = inspect_dataset(TEST_DIR, remove_bad=False)

 TRAIN:
 Cleaning Report:
  Total images : 2622
  Corrupted    : 0
  Duplicates   : 34
  Too small    : 0
  Blurry       : 0
  Too dark     : 0
  Too bright   : 0
  Wrong ext    : 0

   Total issues : 34

 TEST:
 Cleaning Report:
  Total images : 659
  Corrupted    : 0
  Duplicates   : 3
  Too small    : 0
  Blurry       : 0
  Too dark     : 0
  Too bright   : 0
  Wrong ext    : 0

   Total issues : 3


## Cell 4 - نقل الصور

In [6]:
confirm = input('Move bad images? Type YES to confirm: ')
if confirm.strip().upper() == 'YES':
    print('\n📦 Moving bad images...')
    moved = 0
    errors = 0
    for fpath, reason in bad_train + bad_test:
        try:
            bad_dir = os.path.join(os.path.dirname(fpath), '..', '..', 'bad_images')
            os.makedirs(bad_dir, exist_ok=True)
            shutil.move(fpath, os.path.join(bad_dir, os.path.basename(fpath)))
            moved += 1
        except Exception:
            errors += 1
    print(f'  ✅ Moved  : {moved} images')
    print(f'  ❌ Failed : {errors} images')
    print('\n✅ Done! Bad images moved to bad_images folder.')
else:
    print('❌ Cancelled.')

Move bad images? Type YES to confirm:  yes



📦 Moving bad images...
  ✅ Moved  : 37 images
  ❌ Failed : 0 images

✅ Done! Bad images moved to bad_images folder.


## Cell 5 - Resplit Dataset 80% Train / 20% Test

In [ ]:
import random

# ======================================
# إعدادات التقسيم
# ======================================
DATASET_DIR = r'D:\New folder\data'   # Main dataset folder
TRAIN_RATIO = 0.8         # 80% for training
RANDOM_SEED = 42          # Fixed seed for reproducibility
CLASSES     = ['cataract', 'normal']

random.seed(RANDOM_SEED)

print(' Before split:')
print('=' * 40)
for split in ['train', 'test']:
    for cls in CLASSES:
        path = os.path.join(DATASET_DIR, split, cls)
        if os.path.exists(path):
            print(f'  {split}/{cls}: {len(os.listdir(path))} images')

# Collect all images per class
all_images = {cls: [] for cls in CLASSES}

for split in ['train', 'test']:
    for cls in CLASSES:
        path = os.path.join(DATASET_DIR, split, cls)
        if not os.path.exists(path): continue
        for img in os.listdir(path):
            if os.path.isfile(os.path.join(path, img)):
                all_images[cls].append(os.path.join(path, img))

# Create temp folder
TEMP_DIR = DATASET_DIR + '_temp'
for split in ['train', 'test']:
    for cls in CLASSES:
        os.makedirs(os.path.join(TEMP_DIR, split, cls), exist_ok=True)

# Split and copy images
stats = {}
for cls in CLASSES:
    imgs = all_images[cls].copy()
    random.shuffle(imgs)
    split_idx = int(len(imgs) * TRAIN_RATIO)
    train_imgs = imgs[:split_idx]
    test_imgs  = imgs[split_idx:]

    for img_path in train_imgs:
        dst = os.path.join(TEMP_DIR, 'train', cls, os.path.basename(img_path))
        shutil.copy2(img_path, dst)

    for img_path in test_imgs:
        dst = os.path.join(TEMP_DIR, 'test', cls, os.path.basename(img_path))
        shutil.copy2(img_path, dst)

    stats[cls] = {'train': len(train_imgs), 'test': len(test_imgs), 'total': len(imgs)}

# Replace old dataset with new split
for split in ['train', 'test']:
    for cls in CLASSES:
        old_path = os.path.join(DATASET_DIR, split, cls)
        if os.path.exists(old_path):
            shutil.rmtree(old_path)
        shutil.copytree(
            os.path.join(TEMP_DIR, split, cls),
            old_path
        )

shutil.rmtree(TEMP_DIR)

print('\nAfter split:')
print('=' * 40)
for cls in CLASSES:
    print(f'\n  {cls} (total: {stats[cls]["total"]} images):')
    print(f'    train : {stats[cls]["train"]} images (80%)')
    print(f'    test  : {stats[cls]["test"]}  images (20%)')

📊 Before split:
  train/cataract: 1411 images
  train/normal: 1177 images
  test/cataract: 358 images
  test/normal: 298 images


## Cell 6 - Final Dataset Summary

In [7]:
print(' Final Dataset Summary:')
print('=' * 40)
total_train = 0
total_test  = 0
for split in ['train', 'test']:
    path = TRAIN_DIR if split == 'train' else TEST_DIR
    print(f'\n{split.upper()}:')
    for cls in sorted(os.listdir(path)):
        cls_path = os.path.join(path, cls)
        if os.path.isdir(cls_path):
            count = len(os.listdir(cls_path))
            print(f'    {cls}: {count} images')
            if split == 'train': total_train += count
            else: total_test += count

print(f'\n  Total Train : {total_train} images')
print(f'  Total Test  : {total_test}  images')


📊 الداتا النهائية جاهزة للتدريب:

📂 TRAIN:
   cataract: 1431 images
   normal: 1191 images

📂 TEST:
   cataract: 359 images
   normal: 300 images

   Total Train : 2622 images
   Total Test  : 659 images

✅ الداتا جاهزة! افتح 2_model_training.ipynb للتدريب 🚀
